# LiveSQLBench 角色 SQL 数据集构建（SELECT-only）

本笔记本演示如何基于 LiveSQLBench 数据生成带角色约束的 Text-to-SQL 数据集，当前暂时仅支持 SELECT 语句，以便与角色权限模型对齐。

## 使用说明

- 建议先运行 `pip install -r requirements.txt` 并配置好 OpenAI/DeepSeek 等模型的 API Key。
- LiveSQLBench 数据应解压到 `data/livesqlbench-base-lite-sqlite`（或相应版本）目录。
- 默认会尝试复用 `outputs/livesql_data/` 下最近一次生成的角色分配结果，只有在没有缓存时才会触发 LLM 生成流程。
- 当前仅导出 SELECT 语句，以方便在权限模型扩展前保持一致性。

In [ ]:
import json
import logging
import os
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not PROJECT_ROOT or not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Project root not found. Please run this notebook inside the repository.")

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
LIVESQL_OUTPUT_DIR = OUTPUT_ROOT / "livesql_data"
LIVESQL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.role_parser import ParallelRoleGenerator
from src.processors.livesql_describer import LiveSQLDatabaseDescriber
from src.processors.livesql_role_processor import LiveSQLRoleProcessor
from src.processors.livesql_role_sql_generator import LiveSQLRoleSQLGenerator

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("livesql_notebook")

logger.info("Project root: %s", PROJECT_ROOT)

In [ ]:
def find_latest_file(directory: Path, prefix: str, suffix: str = ".json") -> Optional[Path]:
    """Locate the most recent file with the given prefix in ``directory``."""
    candidates = list(directory.glob(f"{prefix}*{suffix}"))
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


def load_cached_role_assignments(role_path: Optional[Path] = None) -> Optional[Dict[str, Sequence[Dict[str, str]]]]:
    """Load cached role assignments from disk if available."""
    if role_path is None:
        role_path = find_latest_file(LIVESQL_OUTPUT_DIR, "role_assignments_")
    if role_path is None or not role_path.exists():
        logger.warning("Cached role assignments not found. Configure API credentials to regenerate roles.")
        return None
    logger.info("Reusing cached roles: %s", role_path)
    with open(role_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    assignments = data.get("assignments")
    if not assignments:
        logger.error("Role cache has unexpected format: %s", role_path)
        return None
    return assignments


def timestamped_filename(prefix: str, extension: str = "json") -> Path:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return LIVESQL_OUTPUT_DIR / f"{prefix}_{stamp}.{extension}"

In [ ]:
def prepare_role_assignments(
    *,
    describer: LiveSQLDatabaseDescriber,
    db_names: Optional[Iterable[str]] = None,
    reuse_cached: bool = True,
    model: Optional[str] = None,
    api_key: Optional[str] = None,
    n_workers: int = 4,
 ) -> Tuple[Path, Dict[str, Sequence[Dict[str, str]]]]:
    """Ensure role assignments exist and return ``(path, assignments)``."""
    if reuse_cached:
        cached_path = find_latest_file(LIVESQL_OUTPUT_DIR, "role_assignments_")
        if cached_path:
            logger.info("Attempting to reuse role cache: %s", cached_path)
            assignments = load_cached_role_assignments(cached_path)
            if assignments:
                return cached_path, assignments
            logger.warning("Cached role file failed to load; generating roles from scratch.")

    model = model or os.getenv("LIVESQL_ROLE_MODEL", os.getenv("OPENAI_MODEL", "gpt-4o-mini"))
    api_key = api_key or os.getenv("LIVESQL_ROLE_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("API key not found. Set OPENAI_API_KEY or LIVESQL_ROLE_API_KEY to regenerate roles.")

    logger.info("Regenerating role assignments with model=%s", model)
    generator = ParallelRoleGenerator(model=model, api_key=api_key, n_workers=n_workers)
    role_processor = LiveSQLRoleProcessor(generator, describer)
    result = role_processor.process_databases(db_names=db_names)
    assignments = result.get("assignments", {})
    if not assignments:
        raise RuntimeError("Role generation returned no assignments. Inspect model output for issues.")

    saved_path = role_processor.save_role_assignments(result, LIVESQL_OUTPUT_DIR)
    return saved_path, assignments


def run_livesql_pipeline(
    *,
    db_names: Optional[Iterable[str]] = None,
    reuse_cached_roles: bool = True,
    allowed_statement_types: Optional[Iterable[str]] = ("select",),
    save_dataset: bool = True,
    model: Optional[str] = None,
    api_key: Optional[str] = None,
    n_workers: int = 4,
 ) -> Dict[str, object]:
    """Execute the LiveSQL pipeline end-to-end and return artefacts."""
    describer = LiveSQLDatabaseDescriber(PROJECT_ROOT)
    role_path, assignments = prepare_role_assignments(
        describer=describer,
        db_names=db_names,
        reuse_cached=reuse_cached_roles,
        model=model,
        api_key=api_key,
        n_workers=n_workers,
    )

    sql_generator = LiveSQLRoleSQLGenerator(
        PROJECT_ROOT,
        describer=describer,
        output_dir=str(LIVESQL_OUTPUT_DIR),
    )

    normalized_allowed = None if allowed_statement_types is None else {typ.lower() for typ in allowed_statement_types}
    dataset = sql_generator.generate_livesql_role_sql_dataset(
        role_file_path=str(role_path),
        data_source="dev",
        allowed_statement_types=normalized_allowed,
    )

    dataset_path = None
    if dataset and save_dataset:
        dataset_path = sql_generator.save_livesql_dataset(dataset)

    return {
        "dataset": dataset,
        "dataset_path": dataset_path,
        "assignments_path": role_path,
        "assignments": assignments,
        "generator": sql_generator,
    }

## 参数配置

在执行管线前可以根据需要调整数据库筛选、允许的语句类型、是否复用缓存角色以及是否保存生成的数据集。

In [ ]:
# Configure LiveSQL pipeline parameters before running
PIPELINE_CONFIG = {
    "db_names": None,  # Limit to a subset like ("virtual", "geo"), None means all databases
    "reuse_cached_roles": True,  # Prefer cached role assignments when available
    "allowed_statement_types": ("select",),  # Keep only the specified SQL statement types
    "save_dataset": True,  # Persist generated datasets under outputs/livesql_data
    "model": None,  # Optional override for the role generation model
    "api_key": None,  # Optional override for API key; falls back to environment variables
    "n_workers": 4,  # Parallel workers when invoking the LLM role generator
}
PIPELINE_CONFIG

### 执行管线

In [ ]:
artifacts = run_livesql_pipeline(**PIPELINE_CONFIG)
dataset = artifacts.get("dataset", [])
dataset_path = artifacts.get("dataset_path")
assignments_path = artifacts.get("assignments_path")
sql_generator = artifacts.get("generator")
assignments_path_display = str(assignments_path) if assignments_path else "Not generated (cache reused)"
dataset_path_display = str(dataset_path) if dataset_path else "Not saved"
summary_lines = [
    "- Dataset size: **{}**".format(len(dataset)),
    "- Role assignment file: `{}`".format(assignments_path_display),
    "- Exported dataset file: `{}`".format(dataset_path_display),
    "- Total examples observed: {}".format(getattr(sql_generator, "get_last_total_example_count", lambda: "unknown")()),
    "- Examples filtered by statement type: {}".format(getattr(sql_generator, "get_last_filtered_example_count", lambda: "unknown")()),
]
display(Markdown("\n".join(["### Run summary"] + summary_lines)))

### 语句类型与权限统计

In [ ]:
statement_counts: Dict[str, int] = {}
if sql_generator is not None:
    statement_counts = sql_generator.get_last_statement_type_counts()
allowed_count = sum(1 for entry in dataset if entry.get("output") != "Sorry, I cannot answer.")
denied_count = len(dataset) - allowed_count
stats_lines = [
    "- Allowed examples: **{}**".format(allowed_count),
    "- Permission-denied examples: **{}**".format(denied_count),
    "- Statement type histogram: `{}`".format(statement_counts or "{}"),
]
display(Markdown("\n".join(["### Permission stats"] + stats_lines)))

### 数据集分析

In [ ]:
if dataset:
    dataset_df = pd.json_normalize(dataset)
    dataset_df["allowed"] = dataset_df["output"].ne("Sorry, I cannot answer.")
    display(dataset_df.head())
else:
    dataset_df = pd.DataFrame()
    logger.warning("Dataset is empty; analysis outputs are skipped.")

In [ ]:
if not dataset_df.empty:
    role_stats = dataset_df.groupby("role").size().rename("count").sort_values(ascending=False)
    display(Markdown("#### Top 10 roles by coverage"))
    display(role_stats.head(10).to_frame())
    db_stats = dataset_df.groupby("db_id").size().rename("count").sort_values(ascending=False)
    display(Markdown("#### Top 10 databases by coverage"))
    display(db_stats.head(10).to_frame())
    allowance_ratio = dataset_df["allowed"].mean()
    display(Markdown(f"#### Permission pass rate: {allowance_ratio:.2%}"))
else:
    logger.info("No dataset rows available for aggregate statistics.")

In [ ]:
if not dataset_df.empty:
    sample = dataset_df.sample(n=min(3, len(dataset_df)), random_state=42)
    previews = []
    for _, row in sample.iterrows():
        preview_lines = [
            f"**Database**: `{row['db_id']}`",
            f"**Role**: `{row['role']}`",
            f"**Permitted**: {'✅' if row['allowed'] else '⛔️'}",
            "",
            "**User query:**",
            row.get("input", ""),
            "",
            "**Model response / gold SQL:**",
            row.get("output", "")[:2000],
        ]
        previews.append("\n".join(preview_lines))
    display(Markdown("\n\n---\n\n".join(previews)))

### Prompt 摘要预览
展示部分样本的完整指令与压缩指令字段，方便比对官方模板和当前实现是否一致。

### 下一步操作提示
- 若需引入 UPDATE/DELETE 等语句，请在 `allowed_statement_types` 中放开对应类型并确保角色权限模型已扩展。
- 生成的完整 prompt 字段参考官方 LiveSQLBench 模板，如需自定义可扩展 `LiveSQLRoleSQLGenerator`。
- 导出文件位于 `outputs/livesql_data/`，可结合评测脚本或下游任务继续处理。